# Clusters

## All Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import AgglomerativeClustering
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("../data/pokedex_post_eda.csv")
df.info()

## Cluster (HP, Attack, Defense, SP_Attack, SP_defense, Speed):

### Metrics

In [ ]:
colunas_batalha = ['HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed']
X = df[colunas_batalha]

scaler = PowerTransformer(method='yeo-johnson', standardize=True)
X_scaled = scaler.fit_transform(X)

plt.figure(figsize=(12, 8))
plt.title("Dendrograma de Pokémon (Baseado em Status de Batalha)")
plt.xlabel("Pokémons (Índice)")
plt.ylabel("Distância Euclidiana (Dissimilaridade)")

dend = shc.dendrogram(shc.linkage(X_scaled, method='ward'),
                      truncate_mode='lastp',
                      p=30,
                      leaf_rotation=90.,
                      leaf_font_size=10.,
                      show_contracted=True)

plt.axhline(y=17, color='r', linestyle='--') 
plt.show()

In [ ]:
range_n_clusters = [2, 3, 4, 5, 6, 7, 8, 9, 10]
silhouette_avg = []

for num_clusters in range_n_clusters:
    clusterer = AgglomerativeClustering(n_clusters=num_clusters)
    cluster_labels = clusterer.fit_predict(X_scaled)
    
    silhouette_avg.append(silhouette_score(X_scaled, cluster_labels))

plt.figure(figsize=(8, 4))
plt.plot(range_n_clusters, silhouette_avg, 'bx-')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.show()

#### Params

In [ ]:
n_clusters = 7

### Group

In [ ]:
colunas_stats = ['HP', 'Attack', 'Defense', 'SP_Attack', 'SP_Defense', 'Speed']
X = df[colunas_stats]

pipeline = Pipeline([
    ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('clustering', AgglomerativeClustering(n_clusters=n_clusters))
])

df['class'] = pipeline.fit_predict(X)

perfil_medio = df.groupby('class')[colunas_stats].mean()

contagem = df['class'].value_counts()

print("Média dos Status por Cluster:")
print(perfil_medio)
print("\nQuantidade de Pokémon por Cluster:")
print(contagem)

for i in range(n_clusters):
    print(f"\nExemplos do Cluster {i}:")
    print(df[df['class'] == i]['Name'].head(5).values)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(pipeline.named_steps['scaler'].transform(X))

df_plot = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_plot['Cluster'] = df['class']

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='Cluster', palette='viridis', s=60)
plt.title("Visualização dos Clusters de Pokémon (Reduzido via PCA)")
plt.xlabel("Componente Principal 1 (Geralmente reflete o 'Poder Total')")
plt.ylabel("Componente Principal 2 (Geralmente reflete o estilo 'Ofensivo vs Defensivo')")
plt.show()

In [ ]:
cross_cat = pd.crosstab(df['class'], df['Category'], normalize='index') * 100

for i in range(n_clusters):
    top_cat = cross_cat.loc[i].idxmax()
    prop_cat = cross_cat.loc[i].max()
    avg_get = df[df['class'] == i]['Get_Rate'].mean()
    
    n_legendaries = df[(df['class'] == i) & (df['Category'] == 'Legendary')].shape[0]
    n_megas = df[(df['class'] == i) & (df['Mega_Evolution_Flag'].notna())].shape[0]
    n_mythicals = df[(df['class'] == i) & (df['Category'] == 'Mythical')].shape[0]
    n_semi = df[(df['class'] == i) & (df['Category'] == 'Semi-Legendary')].shape[0]
    
    print(f"CLUSTER {i}:")
    print(f" - Taxa de Captura Média: {avg_get:.1f}")
    print(f" - Quantidade de Lendários: {n_legendaries}")
    print(f" - Quantidade de Míticos: {n_mythicals}")
    print(f" - Quantidade de Semi-Lendários: {n_semi}")
    print(f" - Quantidade de megas: {n_megas}")
    print("-" * 30)

In [ ]:
for i in range(n_clusters):
    print(f"--- Cluster {i} Samples ---")
    print(df[df['class'] == i]['Name'].sample(5).values)
    print("-"*30)